In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import pandas as pd
import numpy as np
from tqdm import tqdm 
from sklearn.metrics import classification_report, confusion_matrix
import torch
from snntorch import spikegen

from CNN_module.cnn_engine import CNNDetector
from Transformer_module.transformer_engine import TransformerDetector
from SNN_module.snn_engine import SNNDetector

cnn_checker = CNNDetector()
transformer_checker = TransformerDetector()
snn_checker = SNNDetector()

test = pd.read_csv("data/test_data.csv")
test_extended = pd.read_csv("data/test_extended.csv")
hard_test = pd.read_csv("data/hard_test.csv")
df = pd.concat([test, test_extended, hard_test], ignore_index=True)

emails = df['text'].tolist()
tags = df['label'].values
n = len(df)

cnn_results = []
transformer_results = []
snn_results = []
batch_size = 32 

print("Đang tiến hành suy luận (Inference) trên toàn bộ tập dữ liệu...")
for i in tqdm(range(0, n, batch_size)):
    batch_texts = emails[i : i + batch_size]
    
    batch_cnn_preds = cnn_checker.predict(batch_texts)
    batch_trans_preds = transformer_checker.predict(batch_texts)
    batch_snn_preds = snn_checker.predict(batch_texts)

    cnn_results.extend(batch_cnn_preds)
    transformer_results.extend(batch_trans_preds)
    snn_results.extend(batch_snn_preds)

cnn_binary_preds = (np.array(cnn_results) > 50).astype(int)
trans_binary_preds = (np.array(transformer_results) > 50).astype(int)
snn_binary_preds = (np.array(snn_results) > 50).astype(int)

def print_spam_metrics(y_true, y_pred):
    rep = classification_report(y_true, y_pred, output_dict=True)
    spam_metrics = rep['1']
    accuracy = rep['accuracy']
    
    print("2. CHỈ SỐ ĐÁNH GIÁ (METRICS):")
    print(f"   - Độ chính xác tổng thể (Accuracy) : {accuracy*100:.2f}%")
    print(f"   - Độ chuẩn xác Spam (Precision)    : {spam_metrics['precision']*100:.2f}%")
    print(f"   - Độ thu hồi Spam (Recall)         : {spam_metrics['recall']*100:.2f}%")
    print(f"   - Điểm F1 Spam (F1-Score)          : {spam_metrics['f1-score']:.4f}")

# =====================================================================
# ĐÁNH GIÁ ĐỘ CHÍNH XÁC VÀ BÁO CÁO PHÂN LOẠI 
# =====================================================================

print("\n" + "="*50)
print("I: ĐÁNH GIÁ CHUYÊN SÂU CHO MÔ HÌNH CNN")
print("="*50)
cnn_cm = confusion_matrix(tags, cnn_binary_preds)
print("1. CONFUSION MATRIX:")
print(f"                     Dự đoán: HAM   Dự đoán: SPAM")
print(f"Thực tế là HAM:        {cnn_cm[0][0]:<10}   {cnn_cm[0][1]:<10}")
print(f"Thực tế là SPAM:       {cnn_cm[1][0]:<10}   {cnn_cm[1][1]:<10}")
print()
print_spam_metrics(tags, cnn_binary_preds)

print("\n" + "="*50)
print("II: ĐÁNH GIÁ CHUYÊN SÂU CHO MÔ HÌNH TRANSFORMER")
print("="*50)
trans_cm = confusion_matrix(tags, trans_binary_preds)
print("1. CONFUSION MATRIX:")
print(f"                     Dự đoán: HAM   Dự đoán: SPAM")
print(f"Thực tế là HAM:        {trans_cm[0][0]:<10}    {trans_cm[0][1]:<10}")
print(f"Thực tế là SPAM:       {trans_cm[1][0]:<10}    {trans_cm[1][1]:<10}")
print()
print_spam_metrics(tags, trans_binary_preds)

print("\n" + "="*50)
print("III: ĐÁNH GIÁ CHUYÊN SÂU CHO MÔ HÌNH SNN")
print("="*50)
snn_cm = confusion_matrix(tags, snn_binary_preds)
print("1. CONFUSION MATRIX:")
print(f"                     Dự đoán: HAM   Dự đoán: SPAM")
print(f"Thực tế là HAM:        {snn_cm[0][0]:<10}    {snn_cm[0][1]:<10}")
print(f"Thực tế là SPAM:       {snn_cm[1][0]:<10}    {snn_cm[1][1]:<10}")
print()
print_spam_metrics(tags, snn_binary_preds)

# =====================================================================
# ĐÁNH GIÁ NĂNG LƯỢNG TIÊU THỤ (ENERGY CONSUMPTION)
# =====================================================================

print("\n" + "="*50)
print("IV: BÁO CÁO TIÊU THỤ NĂNG LƯỢNG (Phân tích 1 Email)")
print("="*50)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sample_text = emails[0] 

def measure_macs_native(model_checker, text):
    macs_count = [0]
    def linear_hook(module, inp, out):
        in_features = module.in_features
        out_features = module.out_features
        num_elements = out.numel() // out_features
        macs_count[0] += num_elements * in_features * out_features
        
    def conv1d_hook(module, inp, out):
        batch, out_channels, out_len = out.shape
        macs_count[0] += batch * out_channels * out_len * module.in_channels * module.kernel_size[0]
        
    hooks = []
    for module in model_checker.model.modules():
        if isinstance(module, torch.nn.Linear):
            hooks.append(module.register_forward_hook(linear_hook))
        elif isinstance(module, torch.nn.Conv1d):
            hooks.append(module.register_forward_hook(conv1d_hook))
            
    model_checker.predict([text])
    for h in hooks:
        h.remove()
    return macs_count[0]

energy_snn_pj = 0
energy_cnn_pj = 0
energy_trans_pj = 0

print("1. SNN:")
try:
    snn_model = snn_checker.model
    snn_model.eval()
    vectorizer = snn_checker.vectorizer
    input_sparse = vectorizer.transform([sample_text])
    input_tensor = torch.tensor(input_sparse.toarray(), dtype=torch.float32).to(device)
    
    hidden_spikes_count = [0]
    def lif1_hook(module, inp, out):
        spk = out[0]
        hidden_spikes_count[0] += spk.sum().item()
    hook_handle = snn_model.lif1.register_forward_hook(lif1_hook)
    
    with torch.no_grad():
        spike_data = spikegen.rate(input_tensor, num_steps=20)
        spk_out, _ = snn_model(spike_data)
        avg_input_spikes = spike_data.sum().item()
        avg_hidden_spikes = hidden_spikes_count[0]
        hook_handle.remove()
        
        total_sops = (avg_input_spikes * 128) + (avg_hidden_spikes * 2)
        energy_snn_pj = total_sops * 0.9 
        
        print(f"[SNN] Spiking Neural Network:")
        print(f"   - Số phép cộng (SOPs) : {total_sops:,.0f} phép tính")
        print(f"   - Năng lượng ước tính : {energy_snn_pj:,.2f} pJ")
except Exception as e:
    print(f"[SNN] Lỗi đo đếm: {e}")

print("\n2. CNN:")
try:
    macs_cnn = measure_macs_native(cnn_checker, sample_text)
    energy_cnn_pj = macs_cnn * 4.6 
    print(f"[CNN] Convolutional Neural Network:")
    print(f"   - Số phép nhân (MACs) : {macs_cnn:,.0f} phép tính")
    print(f"   - Năng lượng ước tính : {energy_cnn_pj:,.2f} pJ")
except Exception as e:
    print(f"[CNN] Lỗi đo đếm: {e}")

print("\n3. Transformer:")
try:
    macs_trans = measure_macs_native(transformer_checker, sample_text)
    energy_trans_pj = macs_trans * 4.6
    print(f"[Transformer] Self-Attention Network:")
    print(f"   - Số phép nhân (MACs) : {macs_trans:,.0f} phép tính")
    print(f"   - Năng lượng ước tính : {energy_trans_pj:,.2f} pJ")
except Exception as e:
    print(f"[Transformer] Lỗi đo đếm: {e}")

print("\n" + "="*50)
print("V: SO SÁNH SỰ ĐÁNH ĐỔI (TRADE-OFF)")
print("="*50)
if energy_snn_pj > 0 and energy_cnn_pj > 0 and energy_trans_pj > 0:
    print(f"   >> SNN tiết kiệm năng lượng gấp {(energy_cnn_pj / energy_snn_pj):,.1f} lần so với CNN!")
    print(f"   >> SNN tiết kiệm năng lượng gấp {(energy_trans_pj / energy_snn_pj):,.1f} lần so với Transformer!")
else:
    print("   Không đủ dữ liệu để so sánh năng lượng do có lỗi ở các bước đo đếm trên.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: c:\Users\ADMIN\Downloads\Code\.vscode\SPAM_CLASSIFIER\Transformer_module\my_bert_model
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Đang tiến hành suy luận (Inference) trên toàn bộ tập dữ liệu...


100%|██████████| 454/454 [01:47<00:00,  4.21it/s]


I: ĐÁNH GIÁ CHUYÊN SÂU CHO MÔ HÌNH CNN
1. CONFUSION MATRIX:
                     Dự đoán: HAM   Dự đoán: SPAM
Thực tế là HAM:        7334         92        
Thực tế là SPAM:       60           7022      

2. CHỈ SỐ ĐÁNH GIÁ (METRICS):
   - Độ chính xác tổng thể (Accuracy) : 98.95%
   - Độ chuẩn xác Spam (Precision)    : 98.71%
   - Độ thu hồi Spam (Recall)         : 99.15%
   - Điểm F1 Spam (F1-Score)          : 0.9893

II: ĐÁNH GIÁ CHUYÊN SÂU CHO MÔ HÌNH TRANSFORMER
1. CONFUSION MATRIX:
                     Dự đoán: HAM   Dự đoán: SPAM
Thực tế là HAM:        7339          87        
Thực tế là SPAM:       30            7052      

2. CHỈ SỐ ĐÁNH GIÁ (METRICS):
   - Độ chính xác tổng thể (Accuracy) : 99.19%
   - Độ chuẩn xác Spam (Precision)    : 98.78%
   - Độ thu hồi Spam (Recall)         : 99.58%
   - Điểm F1 Spam (F1-Score)          : 0.9918

III: ĐÁNH GIÁ CHUYÊN SÂU CHO MÔ HÌNH SNN
1. CONFUSION MATRIX:
                     Dự đoán: HAM   Dự đoán: SPAM
Thực tế là HAM:        7333 